# Clase 6 — Machine Learning para Predicción de Zonas de Pesca

**Curso:** Inteligencia Artificial Aplicada a la Producción Pesquera  
**Institución:** UTN Facultad Regional Chubut | PesquerosEnIA  
**Autor:** Ariel Giamportone  
**Fecha:** 2026  
**Licencia:** GPL-3.0

---

**Objetivo:** Construir un modelo de Machine Learning que prediga la probabilidad de captura  
exitosa en distintas zonas de la Plataforma Continental Argentina, usando variables  
oceanográficas como predictores.

**Contenido:**
1. Setup
2. El problema: predicción de zonas de pesca
3. Dataset: mareas históricas con variables ambientales
4. Análisis exploratorio (EDA)
5. Preparación de datos
6. Entrenamiento y comparación de modelos
7. Evaluación del mejor modelo
8. Importancia de variables
9. Aplicación práctica: recomendar zona al capitán
10. Reflexión: limitaciones y ética del modelo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)

print('✓ Librerías cargadas')

## 2. El problema: predicción de zonas de pesca exitosas

**Escenario real:**  
Un capitán de arrastrero debe decidir a qué zona de la PCA dirigirse. Tiene disponibles:
- Pronósticos de variables oceanográficas (SST, clorofila, corrientes)
- Historial de capturas propias y de la flota
- Experiencia personal (no cuantificable directamente)

**¿Puede un modelo de ML ayudar en esa decisión?**

**Variable objetivo:** `captura_exitosa`  
- `1` = marea exitosa (captura ≥ umbral definido por la empresa)  
- `0` = marea no exitosa  

**Predictores:** temperatura superficial, clorofila-a, profundidad media, salinidad,  
velocidad de corriente, mes del año, latitud, longitud.

In [ ]:
# ── Dataset: mareas históricas PCA (1.000 registros) ─────────────────────────
np.random.seed(42)
n_mareas = 1000

# Variables oceanográficas
temperatura_superficie = np.random.normal(10, 4, n_mareas)
clorofila_a = np.abs(np.random.exponential(1.8, n_mareas))
profundidad_media = np.random.uniform(50, 300, n_mareas)
salinidad = np.random.normal(33.5, 1.2, n_mareas)
velocidad_corriente = np.abs(np.random.normal(0.3, 0.15, n_mareas))

# Variables espacio-temporales
mes = np.random.randint(1, 13, n_mareas)
latitud = np.random.uniform(-52, -38, n_mareas)
longitud = np.random.uniform(-62, -48, n_mareas)

# Variable objetivo: captura_exitosa
# La captura es mayor cuando: SST ~8-12°C, clorofila alta, profundidad ~80-200m
prob_exito = (
    np.exp(-((temperatura_superficie - 10) ** 2) / 32) * 0.40 +
    np.clip(clorofila_a / 5, 0, 1) * 0.30 +
    np.exp(-((profundidad_media - 140) ** 2) / 5000) * 0.30
)
captura_exitosa = (prob_exito + np.random.normal(0, 0.15, n_mareas) > 0.40).astype(int)

datos_mareas = pd.DataFrame({
    'temperatura_superficie': temperatura_superficie,
    'clorofila_a': clorofila_a,
    'profundidad_media': profundidad_media,
    'salinidad': salinidad,
    'velocidad_corriente': velocidad_corriente,
    'mes': mes,
    'latitud': latitud,
    'longitud': longitud,
    'captura_exitosa': captura_exitosa
})

print(f'Dataset: {datos_mareas.shape[0]} mareas × {datos_mareas.shape[1]} variables')
print(f'Tasa de éxito: {datos_mareas["captura_exitosa"].mean():.1%}')
datos_mareas.head()

## 4. Análisis Exploratorio de Datos (EDA)

Antes de entrenar cualquier modelo, necesitamos entender la distribución de los datos,
las correlaciones entre variables y cómo se diferencian las mareas exitosas de las no exitosas.

In [ ]:
# ── Estadística descriptiva por clase ─────────────────────────────────────────
print('Estadísticas por resultado de marea:')
print(datos_mareas.groupby('captura_exitosa')[[
    'temperatura_superficie', 'clorofila_a', 'profundidad_media'
]].mean().round(2).rename(index={0: 'No exitosa', 1: 'Exitosa'}))

In [ ]:
# ── Boxplots: variables ambientales por resultado de marea ────────────────────
variables_clave = ['temperatura_superficie', 'clorofila_a', 'profundidad_media', 'salinidad']
labels_viz = ['SST (°C)', 'Clorofila-a (mg/m³)', 'Profundidad (m)', 'Salinidad (PSU)']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (var, label) in enumerate(zip(variables_clave, labels_viz)):
    datos_mareas.boxplot(column=var, by='captura_exitosa', ax=axes[i])
    axes[i].set_title(label, fontsize=11)
    axes[i].set_xlabel('Captura exitosa (0=No, 1=Sí)')
    axes[i].set_ylabel(label)

plt.suptitle('Variables Ambientales según Resultado de la Marea\n(Merluza — PCA)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Mapa de correlaciones ─────────────────────────────────────────────────────
variables_corr = [
    'temperatura_superficie', 'clorofila_a', 'profundidad_media',
    'salinidad', 'velocidad_corriente', 'mes', 'captura_exitosa'
]
labels_corr = ['SST', 'Clorofila', 'Profundidad', 'Salinidad', 'Corriente', 'Mes', 'Captura']

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    datos_mareas[variables_corr].corr(),
    annot=True, fmt='.2f', cmap='RdYlBu', center=0,
    xticklabels=labels_corr, yticklabels=labels_corr,
    ax=ax, square=True
)
ax.set_title('Correlaciones entre variables ambientales y captura exitosa', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Preparación de datos para el modelo

Separamos las variables predictoras (`X`) de la variable objetivo (`y`),  
dividimos en conjuntos de entrenamiento y prueba, y escalamos las features.

In [ ]:
# ── Split entrenamiento / prueba ──────────────────────────────────────────────
features = [
    'temperatura_superficie', 'clorofila_a', 'profundidad_media',
    'salinidad', 'velocidad_corriente', 'mes', 'latitud', 'longitud'
]
target = 'captura_exitosa'

X = datos_mareas[features]
y = datos_mareas[target]

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar features (importante para Regresión Logística)
escalador = StandardScaler()
X_entrenamiento_s = escalador.fit_transform(X_entrenamiento)
X_prueba_s = escalador.transform(X_prueba)

print(f'Entrenamiento: {X_entrenamiento.shape[0]} mareas')
print(f'Prueba:        {X_prueba.shape[0]} mareas')
print(f'Tasa de éxito entrenamiento: {y_entrenamiento.mean():.1%}')
print(f'Tasa de éxito prueba:        {y_prueba.mean():.1%}')

## 6. Entrenamiento y comparación de modelos

Comparamos tres modelos con distinto nivel de complejidad:
- **Regresión Logística:** baseline interpretable, lineal
- **Random Forest:** ensamble de árboles, robusto, interpretable por importancia de variables
- **Gradient Boosting:** ensamble secuencial, generalmente el más preciso

Usamos **validación cruzada (5-fold)** con métrica **AUC-ROC** para evitar sobreajuste.

In [ ]:
# ── Cross-validation con 3 modelos ────────────────────────────────────────────
modelos = {
    'Regresión Logística': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

resultados_cv = {}
print('Validación cruzada (5-fold) — Métrica: AUC-ROC\n')
for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo, X_entrenamiento_s, y_entrenamiento, cv=5, scoring='roc_auc'
    )
    resultados_cv[nombre] = scores
    print(f'{nombre:25s}: AUC = {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
# ── Comparación visual de los modelos ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colores = ['#5C85D6', '#2ECC71', '#E74C3C']

for i, (nombre, scores) in enumerate(resultados_cv.items()):
    ax.bar(i, scores.mean(), color=colores[i], alpha=0.85,
           yerr=scores.std(), capsize=5, edgecolor='white')
    ax.text(i, scores.mean() + 0.005, f'{scores.mean():.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_xticks(range(len(modelos)))
ax.set_xticklabels(list(modelos.keys()), fontsize=11)
ax.set_ylim(0.5, 0.95)
ax.set_ylabel('AUC-ROC (promedio 5-fold)', fontsize=12)
ax.set_title('Comparación de Modelos — Predicción de Zona de Pesca Exitosa\n'
             '(barras de error = desviación estándar entre folds)', fontsize=12)
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Clasificador aleatorio')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Evaluación del mejor modelo: Random Forest

Entrenamos el **Random Forest** sobre el conjunto completo de entrenamiento  
y evaluamos su desempeño en el conjunto de prueba (datos no vistos).

In [ ]:
# ── Entrenamiento final del Random Forest ─────────────────────────────────────
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_entrenamiento_s, y_entrenamiento)

y_pred = modelo_rf.predict(X_prueba_s)
y_prob = modelo_rf.predict_proba(X_prueba_s)[:, 1]

print('Reporte de clasificación — conjunto de prueba:')
print(classification_report(y_prueba, y_pred,
                             target_names=['No exitosa', 'Exitosa']))
print(f'AUC-ROC en test: {roc_auc_score(y_prueba, y_prob):.3f}')

In [ ]:
# ── Curva ROC + Matriz de Confusión ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_prueba, y_prob)
auc_score = roc_auc_score(y_prueba, y_prob)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_score:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio (AUC=0.50)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[0].set_xlabel('Tasa de Falsos Positivos', fontsize=12)
axes[0].set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
axes[0].set_title('Curva ROC — Random Forest\nPredicción de Zona de Pesca Exitosa', fontsize=12)
axes[0].legend(fontsize=11)

# Matriz de confusión
disp = ConfusionMatrixDisplay(
    confusion_matrix(y_prueba, y_pred),
    display_labels=['No exitosa', 'Exitosa']
)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Matriz de Confusión — conjunto de prueba', fontsize=12)

plt.tight_layout()
plt.show()

# Interpretación para el sector
cm = confusion_matrix(y_prueba, y_pred)
falsos_positivos = cm[0, 1]
falsos_negativos = cm[1, 0]
print(f'\nFalsos positivos (modelo dice exitosa, pero no lo fue): {falsos_positivos}')
print(f'  → El capitán fue a una zona que el modelo predijo bien, pero no hubo captura')
print(f'Falsos negativos (modelo dice no exitosa, pero sí lo fue): {falsos_negativos}')
print(f'  → El capitán perdió una oportunidad de pesca que el modelo no detectó')

## 8. Importancia de variables: ¿qué factores explican la captura?

El Random Forest puede medir qué tan útil fue cada variable para clasificar correctamente las mareas. Esto nos da conocimiento valioso sobre la biología y ecología pesquera.

In [ ]:
# ── Importancia de variables ───────────────────────────────────────────────────
labels_features = [
    'SST (°C)', 'Clorofila-a', 'Profundidad (m)',
    'Salinidad', 'Corriente (m/s)', 'Mes', 'Latitud', 'Longitud'
]
importancias = pd.DataFrame({
    'variable': labels_features,
    'importancia': modelo_rf.feature_importances_
}).sort_values('importancia', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colores_bar = ['#E74C3C' if i >= len(importancias) - 3 else '#85C1E9'
               for i in range(len(importancias))]
ax.barh(importancias['variable'], importancias['importancia'],
        color=colores_bar, edgecolor='white')
ax.set_xlabel('Importancia relativa (Random Forest)', fontsize=12)
ax.set_title('¿Qué variables oceanográficas predicen mejor\nlas zonas de pesca exitosas?',
             fontsize=12)
plt.tight_layout()
plt.show()

print('\nTop 3 variables más importantes:')
print(importancias.tail(3)[['variable', 'importancia']].to_string(index=False))
print('\nInterpretación biológica:')
print('  SST y clorofila son los principales indicadores del hábitat de merluza.')
print('  Profundidad define el estrato donde viven los cardúmenes demersal-adultos.')

## 9. Aplicación práctica: el capitán quiere decidir a qué zona ir mañana

Tenemos pronósticos oceanográficos para tres zonas posibles. El modelo calcula la probabilidad de éxito en cada una y recomienda la mejor opción.

In [ ]:
# ── Tres zonas candidatas para mañana (julio) ─────────────────────────────────
zonas_candidatas = pd.DataFrame({
    'temperatura_superficie': [9.2,  14.5,  6.8],
    'clorofila_a':            [3.1,   0.9,  1.2],
    'profundidad_media':      [130,   75,   210],
    'salinidad':              [33.4, 34.1, 33.0],
    'velocidad_corriente':    [0.28,  0.20,  0.45],
    'mes':                    [7,     7,     7],
    'latitud':                [-43.5, -40.8, -46.2],
    'longitud':               [-60.2, -58.5, -59.8]
}, index=[
    'Zona A — Frente a Rawson (~43°S)',
    'Zona B — Norte (40°S, aguas más cálidas)',
    'Zona C — Sur (46°S, zona profunda)'
])

# Predicción
zonas_scaled = escalador.transform(zonas_candidatas)
probabilidades = modelo_rf.predict_proba(zonas_scaled)[:, 1]

resultados_zonas = zonas_candidatas[[
    'temperatura_superficie', 'clorofila_a', 'profundidad_media'
]].copy()
resultados_zonas.columns = ['SST (°C)', 'Clorofila', 'Profundidad (m)']
resultados_zonas['Prob. éxito'] = [f'{p:.0%}' for p in probabilidades]
resultados_zonas['Recomendación'] = ['✅ IR' if p == max(probabilidades) else '—'
                                      for p in probabilidades]

print('Recomendación del modelo para el próximo viaje (julio):')
print(resultados_zonas.to_string())
print()
zona_recomendada = zonas_candidatas.index[probabilidades.argmax()]
print(f'→ Zona recomendada: {zona_recomendada}')
print(f'  Probabilidad de captura exitosa: {max(probabilidades):.0%}')
print()
print('Fundamento biológico:')
print(f'  SST {zonas_candidatas["temperatura_superficie"].iloc[probabilidades.argmax()]:.1f}°C'
      f' — dentro del rango óptimo para merluza (8-12°C)')

## 10. Reflexión: limitaciones y uso ético del modelo

### Limitaciones técnicas

- **Datos de entrenamiento:** el modelo aprende del pasado. Si las condiciones oceanográficas cambian significativamente (El Niño, cambio climático), el modelo puede quedar desactualizado.
- **Variables ausentes:** estado del arte de la flota, datos biológicos de INIDEP, precios de mercado, cuotas disponibles — el modelo no las considera.
- **Validación temporal:** siempre entrenar con datos pasados y evaluar con datos futuros (no mezclar temporalmente).

### El modelo y la experiencia del capitán

El modelo **no reemplaza** al capitán — lo **complementa**:
- El capitán sabe cosas que el modelo no puede capturar (estado del arte, información de otros barcos, intuición acumulada)
- El modelo procesa cientos de variables simultáneamente, algo humanamente imposible
- La mejor decisión combina la recomendación del modelo con el juicio del capitán

### Uso sostenible: la IA no debe intensificar el esfuerzo pesquero

Si el modelo mejora la eficiencia de cada viaje, **no debe usarse para aumentar el número de viajes más allá de las cuotas autorizadas**. El objetivo es hacer más eficiente el esfuerzo permitido, no circumvenir la gestión sostenible del recurso.

---

## Para explorar más

- **PesquerosEnIA — ML/DL para ingenieros pesqueros:** https://github.com/PesquerosEnIA/ML_DL_FisheriesEngineers
- **Repo Ariel (ML operativo):** https://github.com/arielgiamportone/Machine_Learning_for_Sales_and_operations_Planning
- **Kroodsma et al. 2018 (Science):** https://www.science.org/doi/10.1126/science.aao5646
- **Global Fishing Watch — datos de pesca:** https://globalfishingwatch.org/data-download
- **scikit-learn — documentación:** https://scikit-learn.org/stable/
- **Próxima clase (8):** Optimización de rutas y operaciones de flota